# Trader Workflow

*Consolidated handover of the three-notebook pipeline: **01 Trader Metrics**, **02 Trader Controls**, **03 Trader Execution**. Spot crypto on Binance. No shorting, no margin, no leverage. The single question this pipeline exists to answer is whether there is a rule the model can trade that clears fees on data it has never seen.*

*This is a technical handover. It states what each stage does, why it is built that way, and how to run it, and it carries the key code and figures from the three notebooks so a new reader can pick up the repository and continue the search. Read Overview first, then the three walkthroughs in order.*

## 0. Overview

### 0.1 After-Fee Evaluation

The goal is one trading strategy that makes money after fees on data it has never seen. Every stage in these three notebooks serves that test, and nothing is accepted without passing it.

A candidate is scored on a blind final year that is held out from all training and tuning. It must clear three bars at once: beat a coin flip at the real base rate, beat buy and hold, and return more than zero per trade after the round-trip fee of about 0.20 percent. A rule that ranks trades well but still loses to the fee is a NO-GO. The fee is the adversary, not the market.

The search runs across three decision frames so intraday, interday, and longer holding each get a fair test. A frame is a whole system: its own bars, its own label, its own features, scored on its own out-of-sample. The frames are compared, never pooled, because a five-minute label and a four-hour label do not mean the same thing.

### 0.2 Pipeline Overview

The work splits into three notebooks that run in order. Each hands a clean input to the next.

| Notebook | Role | The question it answers | Key outputs |
| --- | --- | --- | --- |
| **01 Trader Metrics** | Signal layer | What is worth measuring on a coin | indicators, entry and exit signals, a ranked shortlist, the decision checklist |
| **02 Trader Controls** | Risk and rules layer | What the account is allowed to do | position caps, stops, the daily loss circuit, the fee model, the four-gate screen, position sizing |
| **03 Trader Execution** | Modeling and search layer | Whether a tradeable edge survives fees | the datasets, the features, the model, and the after-fee scoreboard that returns GO or NO-GO |

Metrics defines the vocabulary of signals. Controls sets the boundaries any strategy must live inside, above all the fee and the loss limits. Execution builds the datasets and the model and runs the honest test. The controls are not a step bolted on at the end. They are the filter the whole search is judged against.

### 0.3 Data Frames

Each frame is built from the same Binance Vision archives into its own dataset, `dataset_5m_allmarket`, `dataset_1h_allmarket`, and `dataset_4h_allmarket`, all stored as Parquet. One builder serves all three and retunes its feature windows, its label horizon, and its liquidity and volatility screens to the frame.

| Frame | Bar | Trade style it targets | Default label |
| --- | --- | --- | --- |
| **5m** | 5 minutes | intraday scalps | a shorter ATR barrier, roughly two hours |
| **1h** | 1 hour | day-to-day swings | +2 / -1 ATR triple barrier, two wall-clock days |
| **4h** | 4 hours | multi-day holds | +2 / -1 ATR triple barrier, two wall-clock days |

The higher-timeframe context shifts with the frame, so a five-minute model still sees the one-hour and four-hour trend, a one-hour model sees the four-hour and daily trend, and a four-hour model sees the daily and weekly trend. That is how each frame gets multi-resolution information without the datasets ever being merged. Every modelling section in Part 3 runs this same 1h, 4h, 5m loop and reports one block per frame.

### 0.4 Hard Rules

Every candidate strategy lives inside a fixed set of controls, summarised here and detailed in Part 2. None is negotiable, and the after-fee test in the last row overrides any in-sample result. The rows marked *revised* carry the adjustments logged below; the rest are unchanged from the charter.

| Control | Setting | Why |
| --- | --- | --- |
| Position cap | 5 percent of equity at entry | one bad name cannot sink the book |
| Default size | half Kelly, a quarter on minimum signals | size to the edge, not the conviction |
| Fat-pitch exception | one position to 10 percent, with reward-to-risk at least 3:1, a named cause, and a written exit | rare, documented, reversible |
| Label geometry *(revised)* | longer-horizon, less fee-punishing triple barrier, replacing the +2 / -1 ATR default | the old label lost before any prediction; fewer round trips cut fee drag |
| Hard stop *(revised)* | ATR-scaled per frame, provisional, replacing the fixed 7 percent | trend protection sized to each coin's volatility, not a flat percent |
| Trailing stop *(revised)* | ATR-scaled per frame, provisional, replacing the fixed 10 percent | lets winners run scaled to volatility; settled by the exit-geometry sweep |
| Regime gate *(new)* | act only when BTC is trending up | deploys the real but drowned cross-sectional signal in the regime where it works |
| Daily circuit | halt new orders if rolling 24-hour drawdown passes 3 percent | stop the bleeding, existing stops stay live |
| Drawdown ramp | below a 5 percent rolling-week loss, cut new size with each further 1 percent | shrink the book, do not just block it |
| Cash floor | keep at least 10 percent in cash | always able to act |
| Position limit | at most 3 new positions per week | forces selectivity |
| Direction | spot only, never short, never margin, never leverage | the mandate; shorting is logged below as research, not policy |
| Averaging down | never | a loser is exited or held, not fed |
| Anchoring | cost basis never enters hold or sell logic | decide on forward value only |
| The bar | nothing ships unless it beats a coin flip, buy-and-hold, and the fee out of sample | the fee is the adversary |

**Adjustments logged (this revision).** Four changes were made to release model potential without adding ruin risk. All are bounded and stay inside the no-short mandate. They are documented here as the intended design; the code-level work lands in Part 3 (the label, exit, and regime sections) and has not yet been rebuilt into the on-disk datasets.

1. **Exit geometry.** The fixed 7 percent hard stop and 10 percent trailing stop become ATR-scaled per frame and stay provisional until the exit-geometry sweep, which fits per-coin trailing stops and a time-decaying take-profit and scores them after fees. The flat percent ignored each coin's volatility and conflicted with the code's existing 5 percent and roughly 8.5 percent ATR stops.
2. **Label.** The default +2 / -1 ATR triple barrier is replaced by a longer-horizon, less fee-punishing geometry with fewer round trips. The old label had negative unconditional expectancy, a base rate near 0.313 against a breakeven near 0.333, so it lost before the model predicted anything.
3. **Regime gate (new control).** Deploy the cross-sectional signal only when BTC is trending up. Relative strength is real and broad but drowned by a negative universe baseline, and a regime filter is the cheapest in-mandate way to test whether it turns positive.
4. **Longer-horizon frame.** Extend the search toward the daily frame alongside 4h, because coarser frames cut the fee count per unit of return, which is the central obstacle.

**Unchanged and kept tight.** The 5 percent position cap, half-Kelly sizing, the 3 percent daily circuit, the drawdown ramp, the 10 percent cash floor, the three-new-per-week cap, no averaging down, no anchoring, and the after-fee bar. These bound survival, not the search. Loosening them would amplify a model that is still at the coin-flip floor, which is how no edge becomes ruin.

**Open research question, not adopted: shorting.** Going long the strong third and short the weak third would monetize the cross-sectional spread directly, even against a negative baseline. It is left out because it breaks the spot-only mandate and imports unbounded loss and liquidation risk. If ever tested, it should be small, market-neutral, unleveraged, and paper-first, still behind the after-fee bar.

### 0.5 Running It

The notebooks run on a MacPorts Python through a project `.venv` kernel. Secrets live in the run environment, never in the repository: the Binance and Alpaca keys, `BINANCE_TESTNET`, and the single money switch `LIVE_TRADING`, which stays false until a strategy has earned the right to trade. No real order is placed while it is false; the agent reads the market or runs against the Binance testnet.

One knob drives the modelling cells. `LOAD_FRAME` selects the active frame, 1h by default. Import Data loads that frame's Parquet into `df` and `feat`, and every modelling cell consumes those two. The multi-frame sections in Part 3, model assessment, stability, edge diagnostics, selectivity, and regime conditioning, loop over 1h, 4h, and 5m on their own and restore the active frame when they finish, so the rest of the notebook is unaffected.

Run order is top to bottom. The code cells carried into this document have their outputs cleared, so run them on the Mac with the `.venv` kernel to regenerate every figure and table. One environment caveat: the repository sits on an exFAT volume that scatters `._` AppleDouble files, which can crash matplotlib; clear them if an import throws a `0xb0` decode error.

## 1. Trader Metrics

The signal layer, from `01-trader-metrics`. A read-only daily scan over public market data that ranks coins on an indicator board, with no keys and no orders. It shares Part 3's data spine: price from the survivorship-complete `data.binance.vision` 1h archives resampled to daily, and the universe from Part 3's point-in-time Stage B screen. The scores here are illustrative, a place to build intuition and eyeball candidates, not a trade trigger. Any signal must clear the after-fee, out-of-sample bar in Part 3 before it means anything.

The aligned baseline is the triple-Supertrend the live bot trails and Part 3 scores after fees: three ATR-channel bands, periods 12/3, 10/1, and 11/2, must agree on an uptrend, gated by EMA-200. The board ranks on it first, so all three chapters share one benchmark, and the same signal becomes the `f_st_` feature family in Part 3.

### 1.1 Environment Setup

The scan runs on Python 3.11 with dependencies pinned in `inputs/requirements.txt` and reinstalled on each fresh kernel: `pandas-ta-classic` for indicators, `pykalman` for smoothing, `plotly` and `matplotlib` for charts. No exchange login or API key is needed, because both the price history and the universe come from Part 3's offline archives through `build_dataset_1h`. `ccxt` is kept only for an optional live top-up of the still-forming day.

Provenance is proven, not asserted. Price comes from the official `data.binance.vision` monthly per-symbol OHLCV zips, exchange-direct and checksummed, covering every USDT pair ever listed including delisted ones. That survivorship-complete source is what removes the universe-level bias the old live top-N-by-volume scan carried. The second cell pulls one month of BTCUSDT bars straight from the archive at 1h, 4h, and 5m to show the source and format.

In [ ]:
import sys, subprocess
from pathlib import Path

# assign Python kernel,
REQUIRED_PY = (3, 11)
if sys.version_info[:2] != REQUIRED_PY: raise RuntimeError(
        f"This notebook needs Python {REQUIRED_PY[0]}.{REQUIRED_PY[1]}, "
        f"but the kernel is {sys.version.split()[0]}. Switch the kernel and re-run.")

# Install packages & versions in requirements.txt
REQUIREMENTS = Path("../inputs/requirements.txt")
if not REQUIREMENTS.exists():
    raise FileNotFoundError(f"{REQUIREMENTS} not found — run from the repo root.")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
     "--break-system-packages", "--disable-pip-version-check",
     "-r", str(REQUIREMENTS)], check=True,)

# import package modules and naming
import warnings, numpy as np, pandas as pd
from datetime import datetime
import ccxt
import pandas_ta_classic as ta
from pykalman import KalmanFilter
import plotly, plotly.graph_objects as go, plotly.io as pio
warnings.filterwarnings("ignore")
pio.renderers.default = "plotly_mimetype+notebook_connected"  

# quick sanity check
print(f"Environment ready  ·  python {sys.version.split()[0]}  ({sys.executable})")
print(f"  ccxt {ccxt.__version__} · pandas {pd.__version__} · "
      f"numpy {np.__version__} · plotly {plotly.__version__}")

In [ ]:
# Demo: how each frame is downloaded, and from where. Every frame sources the SAME official Binance
# public archive at data.binance.vision -- monthly per-symbol OHLCV kline zips, one file per interval,
# exchange-direct and checksummed (NOT a live API). ccxt is only a live top-up for the bar still
# forming. This cell pulls one month at 1h, 4h and 5m to prove the source and format; the full pull is
# the commands in the table.
import io, zipfile, urllib.request, pandas as pd
BINANCE_VISION = "https://data.binance.vision/"
def vision_url(symbol, interval, month):
    return f"{BINANCE_VISION}data/spot/monthly/klines/{symbol}/{interval}/{symbol}-{interval}-{month}.zip"

provenance = pd.DataFrame([
    ("1h  (old)", "acquire_vision.py download --interval 1h", "klines_1h/<SYM>/", "dataset_1h_allmarket.parquet"),
    ("1d  (old)", "flow_data.py --interval 1d --all-market",  "klines/<SYM>/",    "daily_flow.csv"),
    ("4h  (new)", "acquire_vision.py download --interval 4h", "klines_4h/<SYM>/", "dataset_4h_allmarket.parquet"),
    ("5m  (new)", "acquire_vision.py download --interval 5m --symbols <scalp set>", "klines_5m/<SYM>/", "dataset_5m_allmarket.parquet"),
], columns=["frame", "download command", "raw klines on disk", "built dataset"])
print("Survivorship-complete: acquire_vision.py crawl enumerates EVERY symbol ever listed (incl.")
print("delisted), so dead coins are not silently dropped the way the exchangeInfo path is.\n")
print(provenance.to_string(index=False))

print("\nLIVE proof of source + format (one month of BTCUSDT, straight from the archive):")
for interval in ("1h", "4h", "5m"):
    u = vision_url("BTCUSDT", interval, "2024-01")
    try:
        raw = urllib.request.urlopen(u, timeout=15).read()
        z = zipfile.ZipFile(io.BytesIO(raw))
        bars = pd.read_csv(io.BytesIO(z.read(z.namelist()[0])), header=None).iloc[:, :6]
        bars.columns = ["open_time", "open", "high", "low", "close", "volume"]
        print(f"  {interval:3s}: {len(bars):5d} bars, {len(raw)/1024:5.0f} KB  <-  {u}")
    except Exception as e:
        print(f"  {interval:3s}: offline ({e}); full archive lives under inputs/binance-data/klines_{interval}/")

### 1.2 Rank Universe

The engine loads the last 400 daily bars per coin, resampled from the 1h archives by `load_daily`, and computes the indicator board: a Kalman smoother read as close-of-day strength, Bollinger Bands for volatility, Ichimoku spans as a trend map, the Archer MA trend flag, RSI, Choppiness, and the triple-Supertrend baseline. The universe is Part 3's survivorship-complete Stage B screen, not a live volume ranking, and stablecoins are dropped because they are flat by construction and would fail Part 2's volatility gate.

Every screened coin is scored into one table, ranked with the triple-Supertrend first and the illustrative indicator flags second. Coins too new to carry full indicator history drop out rather than show invalid values.

In [ ]:
# Data spine: Chapter Three's survivorship-complete 1h archives (data.binance.vision -- every USDT pair
# ever listed, incl. delisted), resampled to daily here. This REPLACES the old live ccxt top-20-by-volume
# scan, which was survivorship-biased at the universe level (it only ever saw today's big coins, was
# point-in-time blind, and so disagreed with the evaluation universe in Chapter Three). ccxt is kept only
# as an OPTIONAL live top-up for the still-forming day; the universe and the history come from the archives.
import os, sys
for _up in (".", "..", "../.."):                       # make inputs/ importable regardless of cwd
    _cand = os.path.abspath(os.path.join(_up, "inputs"))
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand); break
import build_dataset_1h as bd                           # Chapter Three's loader, Supertrend bands, screen

H1_ROOT    = os.path.join(bd.BINANCE_DATA, "klines_1h") # survivorship-complete 1h archives (~600 coins)
TIMEFRAME  = "1d"          # scan frame; daily bars are resampled from the 1h archives in load_daily()
LIMIT      = 400           # daily bars kept per coin for the indicator board
SCAN_N     = 20            # screened coins shown on the board (a display cap, NOT the universe filter)
LIVE_TOPUP = False         # True = append the forming day's bar from ccxt (needs network); archives are closed bars

def load_daily(symbol, limit=LIMIT):
    """Daily OHLCV for one coin, resampled from the survivorship-complete 1h archives (no live fetch).
    `symbol` may be 'BTC/USDT' or 'BTCUSDT'. Returns a datetime-indexed OHLCV frame of the last `limit`
    closed daily bars -- the same data Chapter Three screens and models, just aggregated to the day."""
    o = bd.load_coin(H1_ROOT, symbol.replace("/", ""))
    if o.empty:
        raise FileNotFoundError(f"no 1h archive for {symbol} under {H1_ROOT}")
    o = o.set_index("datetime").sort_index()
    d = o.resample("1D").agg({"open": "first", "high": "max", "low": "min",
                              "close": "last", "volume": "sum"}).dropna()
    return d.tail(limit)

exchange = getattr(ccxt, "binance")()                  # public client, kept ONLY for the optional live top-up
exchange.set_sandbox_mode(False)
print(f"Data spine: survivorship-complete 1h archives -> {TIMEFRAME} bars  |  {LIMIT} bars/coin  |  "
      f"board top {SCAN_N}  |  live top-up {LIVE_TOPUP}")

In [ ]:
def calculate_indicator(symbol, limit=LIMIT):
    df = load_daily(symbol, limit)                                   # daily OHLCV from the survivorship 1h archives
    if LIVE_TOPUP:                                                   # optional: append the still-forming day (ccxt)
        try:
            b = exchange.fetch_ohlcv(symbol, timeframe="1d", limit=2)[-1]
            df.loc[pd.to_datetime(b[0], unit="ms")] = {"open": b[1], "high": b[2], "low": b[3],
                                                       "close": b[4], "volume": b[5]}
        except Exception:
            pass
    close, low = df["close"].iloc[-1], df["low"].iloc[-1]

    # Kalman threshold & smoother
    kf = KalmanFilter(transition_matrices=[1], observation_matrices=[1],
                      initial_state_mean=0, initial_state_covariance=1,
                      observation_covariance=1, transition_covariance=.01)
    state_means, _ = kf.filter(df["close"].values)
    df["kf_mean"] = state_means
    kalman = df["kf_mean"].iloc[-1]
    above_kalman = bool(low > kalman)

    # Trend: Is EMA-14 leading the Kalman mean?
    df.ta.ema(length=14, append=True)
    ema_cross = bool(df["EMA_14"].iloc[-1] > kalman)

    # Bollinger(14) mean-reversion envelope.
    bb = df.ta.bbands(length=14)
    bbl, bbu = bb["BBL_14_2.0"].iloc[-1], bb["BBU_14_2.0"].iloc[-1]

    # Ichimoku projections.
    ich = df.ta.ichimoku()[1]
    isa_9, isb_26 = ich["ISA_9"].iloc[-1], ich["ISB_26"].iloc[-1]

    # Archer MA trend flag, RSI, Choppiness.
    amat = bool(df.ta.amat()["AMATe_LR_8_21_2"].iloc[-1] == 1)
    rsi = float(df.ta.rsi().iloc[-1])
    chop = round(float(df.ta.chop().iloc[-1]), 2)

    # Candle shape check: any doji / dragonfly / gravestone candles?
    o, h, l, c = df["open"].iloc[-1], df["high"].iloc[-1], df["low"].iloc[-1], df["close"].iloc[-1]
    rng   = h - l
    body  = abs(c - o)
    upper = h - max(o, c)
    lower = min(o, c) - l
    doji       = bool(rng > 0 and body <= 0.10 * rng)
    dragonfly  = bool(doji and lower >= 0.6 * rng and upper <= 0.10 * rng)
    gravestone = bool(doji and upper >= 0.6 * rng and lower <= 0.10 * rng)

    # MACD (12,26,9)
    macd_line   = df["close"].ewm(span=12, adjust=False).mean() - df["close"].ewm(span=26, adjust=False).mean()
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    macd, sig           = macd_line.iloc[-1], signal_line.iloc[-1]
    macd_prev, sig_prev = macd_line.iloc[-2], signal_line.iloc[-2]
    macd_buy  = bool(macd_prev <= sig_prev and macd > sig)
    macd_sell = bool(macd_prev >= sig_prev and macd < sig)
    macd_below_zero = bool(macd < 0)

    # Triple-Supertrend baseline -- Chapter Three's actual rules benchmark (the three bands the live bot
    # trails) plus the EMA-200 filter, so the two chapters share ONE baseline. 3/3 bands up AND price above
    # EMA-200 = long. This, not the indicator votes below, is the aligned benchmark; the votes stay context.
    _ups = np.array([bd._supertrend_band(df, p, m)[0] for p, m in bd.ST_BANDS])
    _ema200 = df["close"].ewm(span=bd.ST_EMA, adjust=True).mean()
    st_agree = int(_ups[:, -1].sum())
    st_buy = bool(st_agree == 3 and close > _ema200.iloc[-1])

    buy  = amat and ema_cross and above_kalman
    sell = (not amat) and (not ema_cross) and (not above_kalman)
    row = dict(Symbol=symbol, ST_buy=st_buy, ST_agree=st_agree, Buy=buy, Sell=sell,
               Close=round(float(close), 4),
               RSI=round(rsi, 2), Chop=chop, AMAT=amat,
               Ichimoku_9=round(float(isa_9), 4), Ichimoku_26=round(float(isb_26), 4),
               EMA_gt_Kalman=ema_cross, Low_gt_Kalman=above_kalman,
               Doji=doji, Dragonfly=dragonfly, Gravestone=gravestone,
               MACD=round(float(macd),4), MACD_Signal=round(float(sig),4),
               MACD_Buy=macd_buy, MACD_Sell=macd_sell, MACD_below_zero=macd_below_zero)

    return df, row

def plot(symbol):
    from plotly.subplots import make_subplots
    df, _ = calculate_indicator(symbol)
    # price (candles + Kalman + EMA-14) on top, a volume histogram beneath, coloured by candle
    # direction and sharing the x-axis -- the OHLCV + volume view used across the chapters.
    vol_col = ["#26a69a" if c >= o else "#ef5350" for o, c in zip(df.open, df.close)]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                        row_heights=[0.78, 0.22])
    fig.add_trace(go.Candlestick(x=df.index, open=df.open, high=df.high, low=df.low,
                                 close=df.close, name=symbol), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df["kf_mean"], name="Kalman",
                             line=dict(color="orange", width=2), opacity=0.7), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df["EMA_14"], name="EMA-14",
                             line=dict(color="purple", width=2), opacity=0.7), row=1, col=1)
    fig.add_trace(go.Bar(x=df.index, y=df["volume"], marker_color=vol_col, name="volume",
                         showlegend=False), row=2, col=1)
    fig.update_layout(title=symbol, xaxis_rangeslider_visible=False)
    fig.update_yaxes(title_text="price", row=1, col=1)
    fig.update_yaxes(title_text="volume", row=2, col=1)
    return fig

def color_boolean(val):
    if val is True:  return "background-color: lightgreen"
    if val is False: return "background-color: pink"
    return "background-color: lightblue"

print("Engine ready.")

In [ ]:
# Universe: PREFER Chapter Three's survivorship-complete, point-in-time-screened set (the Stage B
# profile). A profile counts as survivorship-COMPLETE once it has been run over the full klines panel
# that includes delisted coins -- its survivorship.delisted_share > 0. Until then the newest profile is
# survivor-ONLY and the scan inherits that bias, so this cell prefers a complete profile the moment one
# lands and otherwise falls back to the survivor-only set with a printed caveat. Stablecoins (USDC,
# FDUSD, ...) are dropped: they are liquid and "active" so they pass the profile screen, but they are
# flat by construction (a useless candlestick) and would fail Chapter Two's ATR volatility gate.
import json, glob, profile_panel as pp
# USD/EUR-pegged bases to drop from a price-momentum scan (Chapter Two's ATR floor is the general gate)
_STABLE = {"USDC", "USDT", "FDUSD", "USD1", "TUSD", "DAI", "USDP", "BUSD", "GUSD", "USDD",
           "FRAX", "PYUSD", "AEUR", "EUR", "EURI", "EURT", "XUSD", "USTC", "USDS"}
def _pick_profile():
    """Newest profile whose survivorship shows delisted coins (complete); else the newest profile."""
    for ds in sorted(glob.glob(os.path.join(pp.PROFILE_ROOT, "*", "decision_summary.json")), reverse=True):
        if json.load(open(ds)).get("survivorship", {}).get("delisted_share", 0) > 0:
            return os.path.dirname(ds), True
    return pp.latest_run(), False

_run, _complete = _pick_profile()
if _run is None:
    raise FileNotFoundError("no Stage B profile yet -- run `python inputs/profile_panel.py` (Chapter 3, Stage B)")
_md = pd.read_parquet(os.path.join(_run, "symbol_metadata.parquet"))
_on_disk = sum(1 for d in os.listdir(H1_ROOT) if os.path.isdir(os.path.join(H1_ROOT, d)) and not d.startswith("._"))
_screened = _md[(_md["active"]) & (~_md["illiquid_flag"])].sort_values("quote_volume_median", ascending=False)
universe = [f"{s[:-4]}/USDT" for s in _screened["symbol"]
            if s.endswith("USDT") and s[:-4] not in _STABLE
            and os.path.isdir(os.path.join(H1_ROOT, s))][:SCAN_N]
today = datetime.now().strftime("%Y-%m-%d")
_tag = "survivorship-COMPLETE" if _complete else "survivor-ONLY (delisted coins not yet profiled)"
print(f"{today}: {_tag} universe from profile {os.path.basename(_run)} -- scanning "
      f"{len(universe)} of {len(_screened)} screened coins ({_on_disk} on disk; stablecoins dropped).")
if not _complete:
    print("  note: the raw klines panel already includes delisted coins, but Stage B has not been re-run")
    print("  over it. Re-run `python inputs/profile_panel.py` to upgrade this scan to the survivorship-")
    print("  complete universe; this cell will then prefer that profile automatically.")
universe

In [ ]:
rows = []
for symbol in universe:
    try: rows.append(calculate_indicator(symbol)[1])
    except Exception as e: print(f"skip {symbol}: {type(e).__name__}")
results = pd.DataFrame(rows)
print(f"\nscored {len(results)} pairs | {int(results.Buy.sum())} buys | {int(results.Sell.sum())} sells")
results.head()

### 1.3 Performance Metrics

Background methodology, kept for reference. The board itself computes only the guarded MACD crossover, `MACD_Buy` and `MACD_Sell` with a `MACD_below_zero` filter, and reports it beside the Supertrend baseline as context. The divergence taxonomy and the corroborating-indicator table are design notes, not yet implemented in the board.

MACD tracks momentum as the velocity of a trend, the 12-period EMA against the 26. A crossover above the zero line reads bullish and below reads bearish, with the histogram measuring how fast the lines converge or diverge. Earlier signals from the line cross carry more potential than the zero-line cross but also more false starts, which is why the board guards the crossover with the zero-line filter and treats the whole family as illustrative rather than decision-grade. The after-fee, out-of-sample test that would validate any of it is Part 3.

### 1.4 Entry Points

The entry candidates are the top of the board, ranked Supertrend-first, then by the illustrative long flags: the Archer trend, EMA-14 leading the Kalman mean, and price holding above it. The strongest name is drawn on the survivorship archive's daily candles with the Kalman and EMA-14 trend lines and a volume histogram beneath, and saved as a standalone HTML chart.

In [ ]:
top = (results.sort_values(['ST_buy','Buy','EMA_gt_Kalman','AMAT','RSI'], ascending=[False,False,False,False,True]).reset_index(drop=True).head(10))
bottom = (results.sort_values(['Sell','AMAT','RSI'],ascending=[False,False,False]).reset_index(drop=True).head(10))
print("ranked.")

if top['Buy'].any():
    print("Entry candidates showing bullish trends and Kalman lead prices:")
    for s in top.loc[top['Buy'], 'Symbol']: print(" ", s)
else: print("No long signals showing strongest-ranked names.")

top.style.map(color_boolean)

In [ ]:
from IPython.display import HTML
HTML(plot(top['Symbol'].iloc[0]).to_html(include_plotlyjs="cdn", full_html=False))

In [ ]:
# Save the chart as a standalone, self-contained HTML file you can open in any browser.
# include_plotlyjs=True embeds the library so it works offline. No nbformat / .show() needed.
fig = plot(top['Symbol'].iloc[0])
fig.write_html('../outputs/HTML/chart.html', include_plotlyjs=True, auto_open=False)
print('saved outputs/HTML/chart.html')


### 1.5 Exit Points

Read exits before entries. The exit candidates are the weakest-ranked names, the bottom of the board, flagged short when none of the three long conditions hold and price sits below the Kalman mean.

In [ ]:
if bottom['Sell'].any():
    print("Exit candidates trending down showing prices below Kalman:")
    for s in bottom.loc[bottom['Sell'], 'Symbol']: print(" ", s)
else: print("No short signals today; showing weakest-ranked names.")
bottom.style.map(color_boolean)

### 1.6 Exit Geometry

The exit-geometry view is shared verbatim with Part 3 through `inputs/exit_geometry_viz.py`, so the exit points and the trend around each entry sit beside the ranking tables and cannot drift from the model's picture. It draws the three Supertrend trailing lines and EMA-200 over the survivorship 4h archive, shades the window before each entry, annotates the entry with its Supertrend agreement, EMA-200 side, and RSI, and tracks agreement, RSI, and volume beneath. Read-only here; the after-fee scoring of these exits lives in Part 3.

In [ ]:
# Exit geometry + entry trend context (single source: inputs/exit_geometry_viz.py)
import os, sys
for _up in (".", "..", "../.."):
    _cand = os.path.abspath(os.path.join(_up, "inputs"))
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand); break
import exit_geometry_viz as egv
egv.render("BTC/USDT", frame=4, show=True, save=False)   # display inline; chapter three saves the PNGs

### 1.7 Decision Checklist

The board's long and short flags combine three indicators on the last daily bar: the Archer trend flag, EMA-14 leading the Kalman mean, and price above the Kalman mean. A long fires only when all three agree, a short only when none do. Everything else, MACD, Bollinger, Ichimoku, RSI, Choppiness, and the candle checks, is context, not a vote. The aligned baseline remains the triple-Supertrend, and on the 1h frame the after-fee verdict so far is NO-GO, so a green cell means worth a look, not buy.

The checklist for reading the board: read exits before entries, treat a green cell as a candidate to check against the Part 3 evaluation, size and risk-fence in Part 2 before acting, and widen the timelines, since every snapshot is saved and dated under `outputs/`. The daily signals table is written to CSV on each run.

In [ ]:
stamp = datetime.now().strftime('%Y%m%d')
path = f'../outputs/CSV/DailySignals_{stamp}.csv'
results.to_csv(path, index=False)
print("saved", path)

## 2. Trader Controls

*The risk and rules layer, from `02-trader-controls`. The boundaries every candidate must live inside. Subsections to draft, in order:*
- *2.1 Key Variables*
- *2.2 Volatility Filter*
- *2.3 Four-Gate Screen*
- *2.4 Edge Sizing*
- *2.5 Hold Period*
- *2.6 Stacked Defences*
- *2.7 Signal Journal*
- *2.8 Frozen Harness*

## 3. Trader Execution

*The modeling and search layer, from `03-trader-execution`. Where datasets, features, and model meet the after-fee test, run per frame. Subsections to draft, in order:*
- *3.1 Data Preparation*
- *3.2 Survivorship Pipeline*
- *3.3 Feature Variables*
- *3.4 Entries Exits*
- *3.5 Sharpening Entries*
- *3.6 Training Regime*
- *3.7 Model Tuning*
- *3.8 Model Assessment*
- *3.9 Stability*
- *3.10 Edge Diagnostics*
- *3.11 Selectivity Test*
- *3.12 Regime Conditioning*

*The 0.4 adjustments are implemented in this part: the label in 3.1 Data Preparation and 3.6 Training Regime, the ATR-scaled exits in 3.4 Entries Exits, the BTC regime gate in 3.12 Regime Conditioning, and the longer-horizon frame across every per-frame loop.*

## 4. Shared Method

*To draft: the machinery shared by every section, the multi-frame loop and why frames are never pooled, the after-fee scoreboard and the GO / NO-GO gate, embargoed time-series splits, and the provenance discipline that keeps the picture and the numbers from drifting.*

## 5. Findings

*To draft: the honest state of the search. Where each frame lands against the after-fee bar, the one signal that has shown promise (cross-sectional relative strength), and where the edge search points next.*

## Appendices

*To draft:*
- *A. File Map: which module powers which section*
- *B. Glossary: label, triple barrier, ATR, Supertrend, Kelly, embargo, base rate, RMSEratio, and the rest*
- *C. Environment: MacPorts, the `.venv` kernel, the exFAT caveat, and the environment variables*